# Fixed RBF SVM — significant features, no inner search

This exploratory experiment uses fixed `C=1` and `gamma='scale'`, the same 15-subject LOSO test assignments and the same 9,215 labelled windows. It performs **one SVM fit per outer fold**, plus one final all-subject fit: 16 fits total, without inner splits or parameter searches.

Feature selection remains training-only: exact paired sign tests across participant-level stage changes, BY correction across all 36 candidates at q<0.05, then the agreed redundancy filter. **There is no feature-count cap.** This notebook does not freeze the significant list found using the entire cohort. Original notebooks/results are preserved. No real participant-data SVM training was run locally for this delivery.

## 1. Use the existing private Drive project

Copy **this notebook and `fixed_rbf.py`** into your existing Drive `modelling/` folder. Existing `core.py`, `class_balance.py`, requirements and `data/model_windows.csv` are reused. No new dataset upload, CNN signal bundle, GPU or package installation is required. This sklearn SVC uses the CPU even in a GPU-enabled runtime.

The setup follows your working notebook 05: Drive mount, project path, numerical-version checks and fresh-process imports. It does not depend on `environment.py` or change packages. If switching runtimes reset packages, install the existing baseline requirements and restart that kernel before proceeding. Keep the project/output folders private.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from importlib.metadata import version

# Set this only when the notebook is opened outside the cloned project.
PROJECT_ROOT_OVERRIDE = os.environ.get('EMG_PROJECT_ROOT')
start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in [start, *start.parents]
    if (p / 'project_paths.py').is_file() and (p / 'modelling/core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the cloned project, or set EMG_PROJECT_ROOT to its root.')
sys.path.insert(0, str(PROJECT_ROOT))
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
    raise RuntimeError('Restart this kernel, restore INSTALL_REQUIREMENTS=False, then run from the first cell.')
for package, expected in {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0'}.items():
    actual=version(package)
    print(package,actual)
    if actual!=expected:raise RuntimeError('Install requirements in this environment and restart the kernel.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
probe=subprocess.run([sys.executable,'-c','import numpy, pandas, scipy.sparse, sklearn; print("Fresh-process imports OK")'],capture_output=True,text=True)
print(probe.stdout)
if probe.returncode:raise RuntimeError(probe.stderr)
print('Project:',PROJECT_ROOT)
print('Python:',sys.version.split()[0])


Requirements installation is optional in the setup cell above. Restart the same kernel after installing.


Run from the cloned project or set EMG_PROJECT_ROOT to its root. Restore private inputs using the root README.


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Image, Markdown
from modelling.core import load_data
from modelling.fixed_rbf import FIXED_RBF_CONFIG, MODEL_KEY, run_fixed_rbf

DATA_DIR=PROJECT_ROOT / "modelling/data"
OUTPUT_ROOT=PROJECT_ROOT / "modelling/outputs"
CONFIG=dict(FIXED_RBF_CONFIG)
windows,dictionary,provenance=load_data(DATA_DIR)
assert len(windows)==9215 and windows.subject_id.nunique()==15 and windows.recording_id.nunique()==139, \
    "Expected the real feature bundle, not synthetic test files."
display(pd.DataFrame([{"model":MODEL_KEY,"subjects":windows.subject_id.nunique(),
    "sets":windows.recording_id.nunique(),"windows":len(windows),
    "fits_per_outer_fold":1,"total_fits_including_final":16}]))
display(pd.Series(CONFIG,name="Fixed settings — no search").to_frame())


## 2. Labels, eligibility and class counts

Label **0**: first third of the reviewed active interval. Label **1**: last third. These are **elapsed-time thirds**, adapting the assignment's repetition-count example because repetition markers are unavailable. The middle third and any 500 ms windows crossing a stage boundary are excluded. Hop is 250 ms. Keep both legs/all sets together in every split.

The bundle honors the saved set exclusion (`S009_REC01`), reviewed boundaries and computational/flatline/manual-artifact eligibility rules. Provisional transient/drift flags remain flags rather than automatic exclusions. MVC normalization cannot be performed without MVC references. Preprocessing is offline and can use later samples within the same recording; this does not establish real-time performance.

Earlier whole-cohort statistics and trajectories informed development. Fold-local selection prevents direct fitting on held-out subjects, but does not make this a pristine confirmatory study. The exclusion and boundary reviews also preceded validation. A new cohort would provide a stronger independent test.

### Descriptive class counts

Check the retained early/late counts overall, by participant and leg. They are the same labelled windows as in the earlier models. No inner validation partitions are created for this fixed experiment. The near-balanced classes retain the approved no-resampling/no-class-weighting policy.

In [ ]:
from modelling.class_balance import count_table
BALANCE_OUT=OUTPUT_ROOT / "class_balance_fixed_rbf" / provenance["data_sha256"][:12]
BALANCE_OUT.mkdir(parents=True,exist_ok=True)
for name,keys in [("overall",[]),("by_subject",["subject_id"]),("by_leg",["leg"])]:
    table=count_table(windows,keys)
    table.to_csv(BALANCE_OUT / f"class_balance_{name}.csv",index=False)
    display(Markdown("**"+name.replace("_"," ")+"**"));display(table.round(3))
print("Descriptive counts only. No inner splits or model fitting.")


## 3. Approved fixed configuration and training-only selection

**Outer LOSO:** keep both legs and all recordings from a participant together. Fit on 14 participants and evaluate once on the 15th. No inner splits, grid/random search, feature-count tuning, threshold tuning, or probability calibration. There are 15 evaluation fits and one optional final fit. Runtime is still hardware/data dependent, but the original 112-configuration search is removed.

**Significance filter, fitted separately in each outer training set:**

1. For each feature/set/stage, compute the median over eligible windows; require at least three defined windows per stage and paired early/late stages.
2. Calculate late-minus-early change per paired set. Average changes across sets within each leg, then give both legs equal weight within each participant. Require both legs and at least three contributing participants.
3. Apply the exact two-sided paired **sign test** to nonzero participant changes, using zero tolerance 1e-12. Zeros are excluded from the sign-test denominator and reported. Missing/insufficient comparisons receive p=1 for the fixed correction family.
4. Apply Benjamini–Yekutieli correction across all 36 candidates; retain q<0.05. This tests consistent stage-change direction, not physiological specificity or guaranteed classification value.
5. Rank passing candidates by q ascending, absolute standardized participant change descending, then dictionary order. An undefined standardized change sorts last. Traverse this ranking and reject a candidate if |Spearman rho|≥0.90 with a previously retained feature on training-participant changes. Unestimable redundancy comparisons are flagged and that candidate is skipped. There is **no maximum feature count**.

If no feature survives, stop and export diagnostics; do not silently include nonsignificant features or relax the threshold. Feature count/list can differ across folds. All selection decisions use training data only; the full-cohort significance table is not a precomputed model input list.

**Pipeline:** training-only significance/redundancy filter → training-median imputation → training mean/SD scaling → `SVC(kernel='rbf', C=1, gamma='scale', probability=False, tol=0.001, max_iter=-1, cache_size=128)`. Seed20261018; serial execution and one numerical-library thread. `gamma='scale'` is computed from the current training-transformed inputs, not the held-out subject. There is no class weighting/resampling. Use default SVC predictions and uncalibrated decision scores for AUC.

C and gamma are fixed pragmatic choices, not claimed optima. This is a new exploratory procedure after inspecting earlier model results. It changes both the search strategy and feature-selection rule relative to notebook05; do not attribute any difference solely to removing the search. Do not alter settings after viewing this run's outer results and continue calling them untouched.

## 4. Execute one fixed SVM per held-out participant

Each fold prints its participant ID, macro-F1, number of significant and retained features, and duration. Results, selection p/q tables, training-subject changes, split lists and fitted pipelines are saved after each completed fold in a new timestamped directory. Previous runs are preserved. The final all-subject fit has no new unbiased test score.

In [ ]:
OUT=run_fixed_rbf(DATA_DIR, OUTPUT_ROOT, CONFIG)
print("Completed run:",OUT)
run_manifest=json.loads((OUT / "run_manifest.json").read_text())
assert run_manifest["status"]=="complete"
assert run_manifest["n_outer_folds"]==windows.subject_id.nunique()


## 5. Metrics for Sections 5–6

The primary summary gives each held-out participant equal weight: mean ± **sample SD** across 15 outer folds (`ddof=1`). SD is between-participant variation, not a confidence interval. Macro precision/recall/F1 average the two classes equally; zero denominators use zero and supports are shown. Per-class F1 and Cohen's kappa are included.

ROC-AUC, trapezoidal PR-AUC and **average precision (AP)** use late stage as the positive class. PR-AUC and AP use different integration conventions and are reported separately. RBF SVM uses uncalibrated decision scores; logistic regression and the majority baseline use class-1 probabilities. AUCs are evaluated **within each held-out participant** and summarized across folds, without pooling score scales from different fitted models. Undefined AUCs remain missing and the number of defined folds is shown.

For a constant majority score ROC-AUC is 0.5, AP equals late-stage prevalence, and trapezoidal PR-AUC can be noticeably larger because of endpoint interpolation; that does not indicate discrimination. Pooled window-weighted metrics are secondary. Confusion matrices pool counts across all out-of-fold predictions, so participants with more windows contribute more counts.

In [ ]:
fold_metrics = pd.read_csv(OUT / "results.csv")
summary = pd.read_csv(OUT / "metrics_mean_sd.csv")
summary["mean_plus_minus_sd"] = summary.apply(
    lambda r: f"{r['mean']:.3f} ± {r['sd']:.3f}", axis=1)
display(summary[["metric", "mean_plus_minus_sd", "n_defined", "n_folds"]])
display(fold_metrics[["fold", "test_subject", "accuracy", "precision_macro", "recall_macro",
    "f1_macro", "f1_class_0", "f1_class_1", "cohen_kappa", "roc_auc", "pr_auc", "average_precision"]])
display(pd.read_csv(OUT / "per_class_metrics.csv"))
display(pd.read_csv(OUT / "per_class_mean_sd.csv", header=[0,1], index_col=0))
display(Markdown("**Secondary pooled window metrics:**"))
display(pd.Series(json.loads((OUT / "pooled_window_metrics_secondary.json").read_text())).to_frame("value"))

In [ ]:
for image_name in ["confusion_matrices.png", "roc_pr_by_subject.png",
                   "per_subject_f1.png", "metrics_mean_sd.png", "validation_diagram.png"]:
    display(Image(filename=str(OUT / "figures" / image_name)))

## 6. Selection stability, statistical evidence and bilateral results

`n_significant` reports features passing the training-only BY threshold; `n_features` reports those remaining after redundancy filtering. Each fold saves all 36 p-values, q-values, participant counts, direction counts and reasons for retention/rejection in `feature_selection.csv`. These are training-based selection statistics; they are not independent confirmatory results on the test participant.

Selection frequency is descriptive stability, not feature importance or an ablation. RBF SVM has no single linear coefficient per original feature. Kernel/effective gamma and support-vector counts are exported separately. The held-out per-leg table is descriptive and does not establish that one leg fatigues faster.

In [ ]:
display(pd.read_csv(OUT / "selection_frequency.csv"))
display(fold_metrics[["test_subject","n_significant","n_features","selected_features","C","gamma","seconds"]])
display(pd.read_csv(OUT / "per_subject_leg_metrics.csv"))
display(pd.read_csv(OUT / "paired_leg_f1.csv"))
oof=pd.read_csv(OUT / "out_of_fold_predictions.csv")
assert oof.window_id.is_unique and len(oof)==len(windows)
print("Exactly one held-out prediction for every eligible labelled window.")


## 7. Retain this fixed-configuration result

Outputs are saved under `outputs/svm_rbf_fixed_significant/<run>/`, including a separate `results.csv` with 15 rows. The existing model aggregates and expensive RBF runs are preserved. Use macro-F1, variability and per-subject failures when comparing methods; a fixed model is not guaranteed to outperform a tuned one.

The final pipeline is `final_refit/pipeline.joblib`, containing the fitted feature list, imputer, scaler and SVM. Future participants require the same upstream preprocessing/features, then prediction without refitting any step. Load only trusted joblib files.

The final cell archives this run and descriptive class tables in your private Drive folder. Save the executed notebook too. All metrics are generated by the noninteractive `run_fixed_rbf` implementation, also runnable with `python -m modelling.fixed_rbf --data modelling/data`. No old run is automatically resumed or overwritten.

In [ ]:
import shutil
shutil.copytree(BALANCE_OUT, OUT / "class_balance", dirs_exist_ok=True)
archive=shutil.make_archive(str(OUT.parent / (OUT.name+"_results")), "zip",root_dir=OUT)
print("Results archive:",archive)
print("Save this executed notebook separately. Existing model results are preserved.")
